# ReFactX Knowledge-Graph Generation

This notebook uses the rule-backed API in `api.py` with ReFactX's callback-backed `KnowledgeGraphGeneration`. It does not use Postgres.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path

# This notebook lives at the root of the ReFactX-with-Rules repo (where api.py is).
RULES_DIR = Path.cwd()
assert (RULES_DIR / 'api.py').exists(), (
    'Run this notebook from the ReFactX-with-Rules repo root (it contains api.py).')
os.chdir(RULES_DIR)
sys.path.insert(0, str(RULES_DIR))

# The lookup demo uses the companion ReFactX feat/kg checkout.
REFACTX_DIR = Path(os.environ.get('REFACTX_KG_CHECKOUT', RULES_DIR / 'ReFactX'))
REFACTX_DIR = REFACTX_DIR.expanduser().resolve()
if not (REFACTX_DIR / 'refactx').is_dir():
    raise FileNotFoundError('Set REFACTX_KG_CHECKOUT to the ReFactX feat/kg checkout.')
sys.path.insert(0, str(REFACTX_DIR))

import torch
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoProcessor, AutoModelForImageTextToText, TextStreamer
from transformers import ProcessorMixin

import refactx
from refactx.index import DictIndex
from refactx.generate import ConstrainedLogitsProcessor, ConstrainedStateList, KnowledgeGraphGeneration

from api import entity_index, property_strings, get
import api
import pandas as pd

In [ ]:
MODEL = 'Qwen/Qwen3.5-4B'
NUM_BEAMS = 1
NUM_BATCHES = 1
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto', dtype='bfloat16')
tokenizer = processor
streamer = TextStreamer(tokenizer.tokenizer, skip_prompt=True)

## Build the KG index and callbacks from `api.py`

`entity_index()` supplies the initial entities. `property_strings(entity)` supplies the available relations, and `get(subject, relation)` supplies the objects for each relation.

In [ ]:
def _encode_index_text(text):
    tok = tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer
    return tok.encode(text, add_special_tokens=False)

index = DictIndex()
index.set_tokenizer(tokenizer)
entities = entity_index()
for entity in entities:
    # the initial is important because we will generate
    # something like <kg> <Subject> <pred> <obj>
    index.add(_encode_index_text(f' <{entity}>'))

# These are the callback signatures expected by KnowledgeGraphGeneration.
get_relations = property_strings
get_objects = get

print(f'Loaded {len(entities)} entities into the KG index.')

## Register `KnowledgeGraphGeneration`

The `<kg>` marker activates graph traversal. Each generated path is serialized as `<entity> <relation> <object> .`. `long_chains=True` allows the generated object to become the next subject.

In [ ]:
states = ConstrainedStateList(
    'auto',
    num_beams=NUM_BEAMS,
    num_batches=NUM_BATCHES,
    debug_tokenizer=tokenizer,
)

logits_processor = ConstrainedLogitsProcessor(states=states, tokenizer=tokenizer)
logits_processor.add_pattern(
    '<kg>',
    KnowledgeGraphGeneration,
    index=index,
    get_relations=get_relations,
    get_objects=get_objects,
    long_chains=False,
    eot=' </kg>\n',
    avoid_duplicates=True,
    sentinel=True,
)

In [ ]:
# Improved prompt: interleaves reasoning with <kg> lookups and stops as soon
# as the answer is determined (from the companion feat/kg checkout).
prompt_messages = refactx.load_prompt(str(REFACTX_DIR / 'prompts' / 'prompt_qwen36_angular2_kg_nothink.yaml'))

In [ ]:
def _tokenize(text):
    if isinstance(tokenizer, ProcessorMixin):
        return tokenizer.tokenizer(text, return_tensors='pt')
    return tokenizer(text, return_tensors='pt')

def _decode(ids):
    tok = tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer
    return tok.decode(ids, skip_special_tokens=True)

def ask(question, max_new_tokens=512, thinking=False, **generation_kwargs):
    logits_processor.reset_states()
    prompt = refactx.apply_prompt_template(
        tokenizer, question=question, enable_thinking=thinking, prompt_template=prompt_messages)
    inputs = _tokenize(prompt).to(model.device)
    model.eval()
    started = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            logits_processor=[logits_processor],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.80,
            top_k=20,
            min_p=0.0,
            repetition_penalty=1.0,
            num_beams=NUM_BEAMS,
            num_return_sequences=1,
            use_cache=True,
            streamer=streamer,
            **generation_kwargs,
        )
    state = logits_processor.states[0, 0]
    answer = _decode(output[0][inputs.input_ids.shape[1]:])
    paths = []
    for generation in state.generation_history:
        #if hasattr(generation, 'generated_path_metadata'):
        paths.extend(generation.generated_path_metadata)
    print(f'\n--- KG paths ({len(paths)}) ---')
    for path in paths:
        print(path)
    print(f'--- Elapsed: {time.time() - started:.2f}s ---')
    return answer, paths

## Ask questions

Run the following after the setup cells. The prompt should instruct the model to use `<kg>` for graph lookups.

In [ ]:
qa = pd.read_csv(api.QA, keep_default_na=False)

In [ ]:
qa['question'][0]

In [ ]:
qid = 0
print('Question', qa['question'][qid])
answer, paths = ask(qa['question'][qid])
print(answer)

## Example questions

With the improved prompt the model interleaves short reasoning with `<kg>` lookups
and **stops as soon as it can answer** (emitting `<answer>` instead of making more
lookups). Each question below is settled with one or two lookups.

In [ ]:
# Yes/No answered yes after a single lookup
ask('Is it true that Priuli, Gremese e Guarato Group owns Babbo-Toselli SPA?')

In [ ]:
# Yes/No answered no after a single lookup
ask('Is it true that Barberini-Murialdo e figli owns Leoncavallo, Saffi e Gentileschi e figli?')

In [ ]:
# Single entity, one lookup
ask('Who is the ultimate controller of Gremese, Pederiva e Naccari s.r.l.?')

In [ ]:
# Two facts needed (two lookups)
ask('Who are the entities directly connected to Gremese e figli? List at leat 5.',max_new_tokens=2048)

In [ ]:
# Two facts needed (two lookups)
ask('Who are the entities directly connected to Gremese e figli? List at leat 3.')

In [ ]:
# Ask any other question from the benchmark
# ask(qa['question'][3])